# 3. Making schedules compact

[2. Creating a schedule](02_creating_a_schedule.ipynb) turned grouping off to keep things simple.
Left off, a busy participant or chair can end up with just one or two presentations on each of
several different days, instead of having whole days genuinely free. This tutorial turns grouping
back on (it's the default) and shows what it actually changes.

The small conference from tutorials 1-2 is too small to show this well — with only 6
presentations, nobody has enough of them to meaningfully spread out or bunch up. So here we use
`tests/data/several_competing_moderators`, a small dataset shipped with the package's test suite:
4 chairs who each chair 8 presentations, sharing just enough participants with each other that
grouping them well is a genuine search problem rather than something that falls out for free.

## The problem: solving without compactness in mind

First, `optimize_grouping=False`, the same mode tutorial 2 used throughout.

In [1]:
import pathlib

import pandas as pd

from clashless import Presentations, Schedule, SessionTimes, Unavailability

data_dir = pathlib.Path("../../tests/data/several_competing_moderators")
presentations = Presentations(data_dir / "presentations.csv")
unavailability = Unavailability(data_dir / "unavailable.csv")
session_times = SessionTimes(data_dir / "session-start-times.csv")


def active_days_per_chair(schedule):
    merged = schedule.join(presentations.data)
    active_days = merged.groupby("chair")["day"].nunique()
    presentation_count = merged.groupby("chair").size()
    minimum_possible = -(-presentation_count // session_times.n_sessions)
    return pd.DataFrame(
        {
            "presentations": presentation_count,
            "active_days": active_days,
            "minimum_possible": minimum_possible,
        }
    )


fragmented_schedule = Schedule(
    presentations, unavailability, session_times, n_days=6, optimize_grouping=False
).solve()

active_days_per_chair(fragmented_schedule)

,presentations,active_days,minimum_possible
chair,,,
Mod0,8,3,2
Mod1,8,5,2
Mod2,8,4,2
Mod3,8,3,2


`active_days` is well above `minimum_possible` for most of these chairs — each one is spread
across nearly every day of the conference, even though their own presentations could fit into far
fewer days.

## Turning on compactness

`optimize_grouping=True` is the default — tutorial 2 switched it off deliberately. Solving the
*same* data again with defaults tells `solve()` to also minimize, for every participant and
chair, how many distinct days they need (maximizing whole days off).

This is a genuine optimization search rather than "stop at the first valid schedule," so it can
take a while on larger data — best-effort within a time budget, covered below.

In [2]:
compact_schedule = Schedule(
    presentations, unavailability, session_times, n_days=6
).solve()

active_days_per_chair(compact_schedule)

,presentations,active_days,minimum_possible
chair,,,
Mod0,8,2,2
Mod1,8,2,2
Mod2,8,2,2
Mod3,8,2,2


`active_days` should now match `minimum_possible` exactly for every chair — the same people,
the same hard constraints, just arranged more thoughtfully.

## How it works, briefly

`solve()` optimizes two things, in priority order:

1. **Fewest active days** for every participant/chair — the main goal above.
2. **Tightest back-to-back sessions**, as a tiebreaker, on the days they *are* needed — so a
   handful of presentations land in consecutive sessions rather than scattered with gaps between
   them.

The second only ever breaks ties in the first — it can't make someone use an extra day just to
save a gap.

## Trading time for quality

Proving a schedule can't be grouped any better can take arbitrarily long on a big conference, so
`solve()` is best-effort within a time budget: `max_solve_seconds` (default 30). It always still
returns a schedule satisfying every hard constraint - a short budget can only mean a *less
compact* result, never an invalid one.

Our dataset above is a good example of this: easy to satisfy (feasibility is found in well under
a second), but genuinely hard to *optimize* perfectly given only a second or two.

In [3]:
import time

for budget in (1.0, 30.0):
    start = time.perf_counter()
    Schedule(
        presentations, unavailability, session_times, n_days=6, max_solve_seconds=budget
    ).solve()
    print(f"max_solve_seconds={budget}: took {time.perf_counter() - start:.1f}s")

max_solve_seconds=1.0: took 1.2s


max_solve_seconds=30.0: took 30.3s


## Adjusting the weights

Both goals are really one weighted objective: `active_day_weight` (default `n_days * n_sessions`,
large enough to always dominate) and `spread_weight` (default `1`). Cranking `spread_weight` up
far enough inverts the priority - the solver will happily use *more* active days if that's what it
takes to eliminate gaps entirely.

Here's a small example: one chair's 4 presentations, with 4 days available at 2 sessions/day.

In [4]:
weights_dir = pathlib.Path("../../tests/data/grouped_into_fewest_days")
weights_presentations = Presentations(weights_dir / "presentations.csv")
weights_unavailability = Unavailability(weights_dir / "unavailable.csv")
weights_session_times = SessionTimes(weights_dir / "session-start-times.csv")
nora_ids = ["p1", "p2", "p3", "p4"]

default_result = Schedule(
    weights_presentations, weights_unavailability, weights_session_times, n_days=4
).solve()
default_days = default_result.loc[nora_ids, "day"].nunique()
print("default weights    - active days used:", default_days)

spread_first_result = Schedule(
    weights_presentations,
    weights_unavailability,
    weights_session_times,
    n_days=4,
    active_day_weight=1,
    spread_weight=1000,
).solve()
active_days = spread_first_result.loc[nora_ids, "day"].nunique()
print("spread_weight=1000 - active days used:", active_days)

default weights    - active days used: 2


spread_weight=1000 - active days used: 4


With `spread_weight` cranked up, the same presentations spread across all 4 available days -
worse for active days, but every one of them lands entirely alone, with nothing to be spread
across.

## A note on scale

Everything above used a small dataset on purpose. At real conference scale - `tests/data/
large_synthetic` has ~290 presentations - the grouping objective's bookkeeping (tracking which
days and sessions every participant/chair is active on) makes the model large enough that CP-SAT
can spend most of a 30s budget just on that overhead, without reliably beating what a plain
feasibility-only solve already happens to find. That plain solve isn't a bad starting point either
- the way slots are numbered means it tends to fill days in order, which is itself fairly compact
by accident.

In other words: at this scale, `optimize_grouping=True` with the default time budget is not
guaranteed to beat `optimize_grouping=False` - it's a genuinely hard search problem, and 30 seconds
may not be enough time to out-search a lucky accident. If your conference is this large, try a much
larger `max_solve_seconds` and compare against `optimize_grouping=False` on your own data before
relying on the result - and if grouping quality matters more than a fast turnaround, it's worth the
wait.

## Wrap-up

This three-part series covered:

1. [Preparing your data](01_preparing_your_data.ipynb) - the three input tables and their rules.
2. [Creating a schedule](02_creating_a_schedule.ipynb) - solving, understanding the result,
   merging it with your data, visualizing, and exporting it.
3. Making schedules compact (this notebook) - `optimize_grouping`, `max_solve_seconds`, and
   `active_day_weight`/`spread_weight`, all optional and all defaulting to sensible values, so
   everything from tutorial 2 keeps working unchanged if you never touch them - and a note on
   where the default settings currently stop being enough.